# 19 — M10 and M15 on **PPC** → the SDL's AI models plan the experiments

**What this does, in plain words.** Same machine as notebook 18, for the two lowest design grades:
5 designs and 3 IS trials per grade, trials picked to teach the slump model most, the single best
next experiment per grade, and a results log that makes both AI models learn when re-run.

**What the standard allows for these grades** (read from `is.456.2000.pdf`):

| standard | clause / table | requirement | for this project |
|---|---|---|---|
| IS 456:2000 | cl 6.1.2 + Table 5 col 8 | minimum grade for **reinforced** concrete is M20 (mild) and higher for harsher exposure | M10 / M15 are **not** normal RCC grades |
| IS 456:2000 | Table 5 col 5 + Note 2 | plain concrete: no minimum grade under mild exposure; M15 under moderate; M20 under severe | M10 = **PCC, mild** only; M15 = PCC, mild or moderate |
| IS 456:2000 | cl 6.1.3 | lower grades may be used for plain concrete, lean concrete, simple foundations, masonry-wall foundations, simple or temporary RCC | engineer decides if an order falls here |
| IS 456:2000 | Table 5 (PCC, mild) | min cement 220 kg/m³, max free w/c 0.60 | applied by the Layer 0 gate |
| IS 456:2000 | Table 11 | compliance rows printed for **M15** and **M20 or above** only | **M10 acceptance criterion: NOT VERIFIED FROM PRIMARY STANDARD SOURCE** — engineer to set |
| IS 10262:2019 | Tables 1, 2 | X and assumed S start at M10 | design route applies to both grades |

**Order inputs used here** `[ASSUMED]`: PCC, mild exposure, 130 mm slump (the target recorded with
the dataset's own M10 / M15 batches), pumped. Change them in section 1 and re-run.

**AI models.** They learned from PPC + crushed-sand families **M20–M40** (binder 327–450 kg/m³).
For M10/M15 the IS water demand at 130 mm and the Table 5 w/c 0.60 limit already push the binder
to about 310–330 kg/m³, so the cheapest legal mixes sit at the lower edge of what the models know.
Section 3b checks the models against the 5 real M10/M15 batches in the dataset (not used in training).

Tags: `[IS]` standard · `[EI]` engineering interpretation · `[PC]` project choice ·
`[ML]` model · `[BO]` experiment-design method (not an IS requirement) · `[ASSUMED]` declared.
Every mix here is **COMPUTATIONALLY COMPLIANT** only. Run notebook 18 once first (it writes
`p1_ppc_loop.py`, used here).

In [ ]:
import sys as _sys
for _m in ("is_rules", "layer0", "workability_residual", "strength_residual",
           "screen", "selection", "loop", "p1_ppc_loop"):
    _sys.modules.pop(_m, None)
import math, json, warnings
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, openpyxl
import is_rules as IS
import layer0 as L0
import workability_residual as WR
import strength_residual as SR
import screen as SC
import selection as SEL
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
F3 = lambda x: f"{x:9.3f}"

## 0. The learning helpers (from `p1_ppc_loop.py`, written by notebook 18)

In [ ]:
# helpers shared with notebook 18 -- run 18 once first
from p1_ppc_loop import *

In [ ]:
# EXPORT: skip -- fixtures for the update maths
_w = joblib.load("workability_residual_model.joblib")
_tr = pd.read_csv("04_workability_dataset.csv")
_tr = _tr[_tr["design_grp"].astype(str).str.contains("[A-Za-z]", na=False)].head(3).copy()
_mu0, _ = WR.predict_slump(_w, _tr, kind="mean")
_same = condition(_w, _tr.iloc[:0], [], [], "none")
assert np.allclose(WR.predict_slump(_same, _tr, kind="mean")[0], _mu0), "empty update changed the model"
_red = WR.water_reduction_pct(_tr["sp_pct_binder"].to_numpy(float), _w["water_fit"])
_base = np.asarray(WR.is_slump_baseline(_tr["water_kg_m3"], _tr["msa"], water_reduction_pct=_red))
_fake = _mu0[:1] + 40.0                                   # pretend the lab measured 40 mm more
_upd = condition(_w, _tr.iloc[:1], _fake - _base[:1], ["FIXTURE"], "fixture")
_mu1 = WR.predict_slump(_upd, _tr.iloc[:1], kind="mean")[0]
assert _mu1[0] > _mu0[0] + 1.0, "a higher measured slump must pull the prediction up"
_i = information(_w, _tr.iloc[:1], _tr)
assert _i["bits"] >= 0 and _i["sd_after"] <= _i["sd_before"] + 1e-9
print(f"fixtures OK: empty update = no change; +40 mm reading moves prediction "
      f"{_mu0[0]:.3f} -> {_mu1[0]:.3f} mm; information >= 0, sd never rises")

## 1. The orders

| input | M10 | M15 | classification |
|---|---|---|---|
| grade | M10 | M15 | ENGINEER INPUT |
| use | PCC | PCC | IS 456 Table 5: RCC needs ≥ M20 (cl 6.1.3 exceptions are the engineer's call) |
| exposure | mild | mild | **[ASSUMED]**; mild is the only Table 5 exposure that allows M10 PCC |
| target slump | 130 mm | 130 mm | **[ASSUMED]** target recorded with the dataset's M10/M15 batches |
| pumped | yes | yes | **[ASSUMED]** as the training plant's M20 family |
| cement | PPC | PPC | PROJECT CHOICE (user) |
| PPC, fly ash, sand, coarse aggregate | `Plant D` M20 family | same | MATERIAL-SPECIFIC — the lowest-grade family the models learned from |
| concrete temperature | family median | same | PLANT-SPECIFIC |
| admixture range / fly ash SG | 0.621–1.476 % / 2.20 | same | PLANT-OBSERVED / **[ASSUMED]** (as notebooks 12, 18) |

In [ ]:
GRADES = ["M10", "M15"]
BEST_GRADES = GRADES
LOG_PATH = "M10_M15_PPC_results_log.csv"
SLUMP_MM = 130.0                      # [ASSUMED]
EXPOSURE = IS.Exposure.MILD           # [ASSUMED]
USE = IS.UseClass.PCC
FLYASH_SG = 2.20                      # [ASSUMED] IS 10262 Annex B example value
ADMIXTURE_RANGE_PCT = (0.621, 1.476)  # [PLANT-OBSERVED]
MATERIAL_OVERRIDES = {}               # the lab's own PPC lot / sand values, if different
MOISTURE_PCT = {"sand": None, "ca": None}
ASSUMED = ["PCC", "mild exposure", "target slump 130 mm", "pumped", "fly ash SG 2.20",
           "admixture range plant-observed", "moisture = training-plant median until measured",
           "M10 acceptance criterion not printed in IS 456 Table 11 -- engineer to set"]
README_TITLE = "Notebook 19 -- M10 and M15 on PPC, AI-planned experiments"
README_LINES = [
    "M10 and M15 on PPC, planned by the SDL's AI models (slump + strength). Plain concrete (PCC), mild exposure, 130 mm slump -- all ASSUMED, change in notebook 19 section 1.",
    "IS 456 Table 5: RCC needs M20 or more, so M10/M15 are PCC (cl 6.1.3 exceptions are the engineer's call). IS 456 Table 11 has no M10 row: the engineer sets M10 acceptance.",
    "Materials = the PPC, fly ash and 100 % crushed sand of the M20 family the AI models learned from (Plant D).",
    "Sheet 'Best next' = the one mix per grade to cast first. 'Trials' = IS 10262 cl 5.8 trials per grade, picked to teach the slump model most.",
    "At 130 mm, IS water demand + Table 5 w/c 0.60 set the binder, not strength: expect these mixes to be well above M10/M15 strength.",
    "After testing: fill the yellow cells in 'Results log', save as M10_M15_PPC_results_log.csv in the folder, re-run notebook 19.",
    "Every mix is COMPUTATIONALLY COMPLIANT only. Engineer sign-off and physical trials decide."]

# [IS] IS 456 Table 5 -- which use / exposure is legal for these grades (minimum grade, col 5 and col 8)
_leg = []
for _e in ("mild", "moderate", "severe", "very_severe", "extreme"):
    for _u in ("pcc", "rcc"):
        _mg = IS.IS456_TABLE5[_e][_u][2][0]
        _leg.append({"exposure": _e, "use": _u, "min_grade": _mg or "not specified (Note 2)",
                     **{g: (_mg is None) or IS.grade_to_fck(g) >= IS.grade_to_fck(_mg) for g in GRADES}})
print("IS 456 Table 5 -- is the grade allowed?")
print(pd.DataFrame(_leg).to_string(index=False))

wmodel0 = joblib.load("workability_residual_model.joblib")
smodel0 = SC.load_strength_model()
ORDERS = SC.plant_orders(flyash_sg=FLYASH_SG, admixture_range_pct=ADMIXTURE_RANGE_PCT)
_fam = next(o for o in ORDERS if "req" in o and o["req"].grade == "M20")   # lowest-grade PPC family
STATE, REQ, FAM = {}, {}, {}
for g in GRADES:
    st = replace(_fam["state"], **MATERIAL_OVERRIDES) if MATERIAL_OVERRIDES else _fam["state"]
    STATE[g], FAM[g] = st, _fam
    REQ[g] = L0.Requirement(grade=g, target_slump_mm=SLUMP_MM, exposure=EXPOSURE, use=USE,
                            pumpable=bool(_fam["pumped"]), pumpable_reduction_frac=1.0,
                            concrete_temp_c=_fam["req"].concrete_temp_c)
    tm = IS.target_mean_strength(g)
    print(f"{g}: {USE.name}, {EXPOSURE.name}, slump {SLUMP_MM:.3f} mm, temp {REQ[g].concrete_temp_c:.3f} C; "
          f"{st.cement_type} (28-d {st.cement_28d_strength:.3f} MPa) from {_fam['family']}; target mean "
          f"{tm['target_mean']:.3f} MPa (IS 10262 cl 4.2: fck + 1.65 S = {tm['fck_plus_1_65S']:.3f}, fck + X = {tm['fck_plus_X']:.3f}, S {tm['S']:.3f})")

## 2. Learn from the lab (runs automatically once results are in the log)

The lab writes results into `M10_M15_PPC_results_log.csv`. Only rows whose `outcome` is
`measured` (or `measured_workability` / `measured_strength`) are used — a rejection without a
cast is not data. One physical batch = one row, however many cubes it made. A 28-day sample
is the mean of 3 cubes and is thrown out if a cube is more than 15 % off the mean
**[IS] IS 456 cl 15.3 / 15.4**.

In [ ]:
LOG = read_log(LOG_PATH)          # this notebook's own log (plans are written here)
LOG_ALL = read_all_logs()          # every PPC log in the folder -- the models learn from all of them
wmodel, smodel, UPD = update_models(wmodel0, smodel0, LOG_ALL)
print(f"lab rows in all PPC logs: {len(LOG_ALL)}; measured slump batches used: {UPD['n_slump_batches']}; "
      f"measured strength samples used: {UPD['n_strength_batches']}")
for n in UPD["notes"]:
    print("  note:", n)
if UPD["slump_errors"]:
    e = pd.DataFrame(UPD["slump_errors"]); e["error_mm"] = e["predicted_mm"] - e["measured_mm"]
    print("\nslump: what the model said BEFORE the cast vs the lab"); print(e.round(3).to_string(index=False))
if UPD["strength_errors"]:
    e = pd.DataFrame(UPD["strength_errors"]); e["error_mpa"] = e["predicted_mpa"] - e["measured_mpa"]
    print("\n28-day: what the model said BEFORE the cast vs the lab"); print(e.round(3).to_string(index=False))
if not (UPD["n_slump_batches"] or UPD["n_strength_batches"]):
    print("no lab results yet -> the models are exactly as notebooks 10 / 10b left them")
else:
    joblib.dump(wmodel, "19_workability_model_updated.joblib")
    if smodel is not None:
        joblib.dump(smodel, "19_strength_model_updated.joblib")
    print("wrote 19_workability_model_updated.joblib / 19_strength_model_updated.joblib "
          "(the notebook 10 / 10b files are not touched)")

# the box the slump model has seen, widened by any lab batch it has now learned from
_train = pd.read_csv("04_workability_dataset.csv")
_train = _train[_train["design_grp"].astype(str).str.contains("[A-Za-z]", na=False)]
_wl, _, _ = measured_rows(LOG_ALL)
if len(_wl):
    _add = _wl.copy()
    for f in wmodel["features"]:
        _add[f] = pd.to_numeric(_add[f], errors="coerce")
    _train = pd.concat([_train, _add[wmodel["features"] + ["slump_mm"]]], ignore_index=True)
SUP = SC.training_support(_train, wmodel["features"])
SLUMP_SEEN = (float(_train["slump_mm"].min()), float(_train["slump_mm"].max()))

## 3. Can the AI models be used for these orders? (domain check)

In [ ]:
dom = []
for g in GRADES:
    st, rq = STATE[g], REQ[g]
    dom += [
        {"grade": g, "check": "cement type", "value": st.cement_type,
         "model_range": smodel["cement_type"], "inside": st.cement_type == smodel["cement_type"]},
        {"grade": g, "check": "crushed-sand share", "value": 1.0 if st.sand_type == "crushed" else 0.0,
         "model_range": f"{SUP['msand_frac'][0]:.3f}-{SUP['msand_frac'][1]:.3f}",
         "inside": SUP["msand_frac"][0] <= (1.0 if st.sand_type == "crushed" else 0.0) <= SUP["msand_frac"][1]},
        {"grade": g, "check": "concrete temperature C", "value": rq.concrete_temp_c,
         "model_range": f"{SUP['concrete_temp'][0]:.3f}-{SUP['concrete_temp'][1]:.3f}",
         "inside": SUP["concrete_temp"][0] <= rq.concrete_temp_c <= SUP["concrete_temp"][1]},
        {"grade": g, "check": "target slump mm", "value": rq.target_slump_mm,
         "model_range": f"{SLUMP_SEEN[0]:.3f}-{SLUMP_SEEN[1]:.3f}",
         "inside": SLUMP_SEEN[0] <= rq.target_slump_mm <= SLUMP_SEEN[1]},
    ]
DOM = pd.DataFrame(dom)
print(DOM.to_string(index=False))
_t = pd.read_csv("04_workability_dataset.csv")
_t = _t[_t["design_grp"].astype(str).str.contains("[A-Za-z]", na=False)]
LOSS = (pd.to_numeric(_t["slump"], errors="coerce") - pd.to_numeric(_t["slump_45min"], errors="coerce")).dropna()
TEST_MIN = pd.to_numeric(_t["time_batch_to_test"], errors="coerce")
print(f"\nThe slump model's data were measured {TEST_MIN.min():.0f}-{TEST_MIN.max():.0f} min after batching "
      f"(median {TEST_MIN.median():.0f}). At that plant the slump then fell by a median {LOSS.median():.3f} mm "
      f"(range {LOSS.min():.3f}-{LOSS.max():.3f}) by 45 min. So a predicted {REQ[GRADES[0]].target_slump_mm:.0f} mm is the "
      f"~15-min slump; if it is needed at 45 min, expect it lower. The 15/45/60-min readings will measure this.")
print("\nPPC + crushed sand + temperature: inside. Binder, fly-ash share and dose are checked per mix")
print("(column inside_training_range).")
if not DOM["inside"].all():
    print("\nOUTSIDE: the slump targets are above every slump the model has seen. The slump at these")
    print("levels comes from the IS 10262 cl 5.3 water relation + the model's correction, i.e. an")
    print("extrapolation in slump level. That is exactly why these trials teach the slump model most:")
    print("each measured batch at 150-180 mm is new information it does not have.")

## 3b. Reality check — the models against the 5 real M10 / M15 batches in the dataset

These 5 batches (2 from Plant A, OPC 53 + fly ash; 3 from Plant B, cement type not recorded) were
**not** used to train the models (one-off groups, quarantined in notebook 10). Comparing them
shows how far the models can be trusted in this low-grade region. Different cement, so this is
a rough check, not a validation. Missing inputs are filled with the training medians and listed.

In [ ]:
_d = pd.read_csv("04_workability_dataset.csv", low_memory=False)
REAL = _d[_d["grade"].isin(GRADES)].copy().reset_index(drop=True)
filled = {}
REAL["msa"] = pd.to_numeric(REAL["msa"], errors="coerce")
if REAL["msa"].isna().any():
    filled["msa"] = int(REAL["msa"].isna().sum()); REAL["msa"] = REAL["msa"].fillna(20.0)
for f in sorted(set(wmodel["features"]) | set(smodel["features"])):
    REAL[f] = pd.to_numeric(REAL.get(f), errors="coerce")
    med = {**wmodel.get("feature_median", {}), **smodel.get("feature_median", {})}.get(f)
    if REAL[f].isna().any():
        filled[f] = int(REAL[f].isna().sum()); REAL[f] = REAL[f].fillna(med)
REAL["slump_pred"], _ = WR.predict_slump(wmodel, REAL, kind="mean")
REAL["fc_pred"], _ = SR.predict_strength_mpa(smodel, REAL, kind="sample")
REAL["fc28_measured"] = REAL[["cube_28d_1", "cube_28d_2", "cube_28d_3"]].apply(pd.to_numeric, errors="coerce").mean(axis=1)
REAL["inside_training_range"] = REAL.apply(lambda r: SC.inside_support(r, SUP), axis=1)
REAL["target_mean"] = [IS.target_mean_strength(g)["target_mean"] for g in REAL["grade"]]
show = REAL[["grade", "plant_lab_id", "cement_type", "binder_kg_m3", "scm_frac", "wb_ratio", "sp_pct_binder",
             "slump_mm", "slump_pred", "fc28_measured", "fc_pred", "target_mean", "inside_training_range"]].copy()
show["plant_lab_id"] = show["plant_lab_id"].astype(str).str[:12]
print(show.round(3).to_string(index=False))
print(f"\nfilled with training medians: {filled}")
print(f"slump error (pred - measured): mean {float((REAL.slump_pred - REAL.slump_mm).mean()):.3f} mm; "
      f"28-d error: mean {float((REAL.fc_pred - REAL.fc28_measured).mean()):.3f} MPa")
print("All 5 are leaner than anything the models learned from (binder 240-300, w/b 0.54-0.65): treat")
print("the models' numbers below the M20 family as guides, and let the lab results correct them.")

## 4. The AI screen — every legal mix, scored by both models

`[IS]` Every candidate passes the IS 456 / IS 10262 gate first (`layer0.py`). Then the slump
model gives P(slump within the IS tolerance) and the strength model gives P(28-day sample ≥
target mean). **SERVE** keeps only mixes inside what the models have seen; **EXPLORE** also
keeps the ones outside (that is where a trial teaches most). Standards limits are identical in both.

In [ ]:
SERVE, EXPL, DESIGNS, RMETA = {}, {}, {}, {}
for g in GRADES:
    SERVE[g], _ = SC.screen(REQ[g], STATE[g], wmodel=wmodel, smodel=smodel, support=SUP, mode="serve")
    EXPL[g], _ = SC.screen(REQ[g], STATE[g], wmodel=wmodel, smodel=smodel, support=SUP, mode="explore")
    ranked, RMETA[g] = SEL.rank_serve(SERVE[g], p_strength_min=0.90, p_slump_min=0.80, front_only=False)
    d = SEL.pick_designs(ranked, 5)
    if len(d):
        d["design_status"] = "meets P_strength >= 0.90 and P_slump >= 0.80"
    else:
        # nothing meets both thresholds: show the 5 closest, clearly marked, never as supply mixes
        near = SEL.price_and_carbon(SERVE[g])
        near = (near.sort_values(["P_strength", "cost_inr_m3"], ascending=[False, True])
                    .drop_duplicates(["binder_kg_m3", "scm_frac", "sp_pct_binder"]).head(5).reset_index(drop=True))
        near.insert(0, "design_rank", range(1, len(near) + 1))
        near["validation_level"] = L0.ValidationLevel.COMPUTATIONALLY_COMPLIANT.name
        near["on_pareto_front"] = False
        near["design_status"] = "NOT SUPPLY-READY: closest mixes, P_strength < 0.90 -- learning only"
        d = near
    DESIGNS[g] = d.assign(grade=g)
    print(f"\n{g}: {len(SERVE[g])} in-range legal mixes, {len(EXPL[g])} incl. out-of-range; "
          f"feasible {RMETA[g].get('n_feasible', 0)}; excluded {RMETA[g].get('excluded')}")
    print(DESIGNS[g][["design_rank", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3",
                      "sp_pct_binder", "wb_ratio", "fc_pred", "P_strength", "slump_pred", "P_slump",
                      "cost_inr_m3", "design_status"]].to_string(index=False, float_format=F3))

## 5. The 3 trials per grade — chosen to teach the slump model

`[IS]` IS 10262 cl 5.8: Trial 1 = the design; Trials 3 and 4 = same water, w/cm about −10 % and
+10 %. Any of the 5 designs may be the Trial 1. `[BO]` We build the trial set around **each** of
them and keep the one whose 3 slump readings teach the slump model the most (bits of
information, worked out before casting), cost breaking ties. "Use set" = all in-range legal mixes
of both grades — the mixes the model will be asked about next.

In [ ]:
def trial_set_for(g, anchor):
    req, st = REQ[g], STATE[g]
    wfit = wmodel["water_reduction_fit"]
    req_c = replace(req, water_reduction_pct=float(WR.water_reduction_pct(float(anchor["dv_sp_pct_binder"]), wfit)))
    ts = L0.trial_set(req_c, st, binder_kg_m3=float(anchor["dv_binder_kg_m3"]),
                      flyash_frac=float(anchor["dv_flyash_frac"]), ggbs_frac=float(anchor["dv_ggbs_frac"]),
                      admixture_pct_of_binder=float(anchor["dv_sp_pct_binder"]))
    ok = [t for t in ts if t["mix"] is not None]
    sc, _ = SC.screen(req, st, wmodel=wmodel, smodel=smodel, support=SUP, mode="score",
                      mixes=[t["mix"] for t in ok])
    sc = SEL.price_and_carbon(sc).assign(trial_no=[t["trial_no"] for t in ok],
                                         trial_role=[t["role"] for t in ok],
                                         trial_note=[t["note"] for t in ok], grade=g)
    return sc, [t["mix"] for t in ok], [t for t in ts if t["mix"] is None]

USE = pd.concat([SERVE[g] for g in GRADES], ignore_index=True)
TRIALS, MIXES, CHOICE = {}, {}, []
for g in GRADES:
    best = None
    for _, a in DESIGNS[g].iterrows():
        sc, mixes, refused = trial_set_for(g, a)
        if len(sc) < 3:
            continue
        inf = information(wmodel, sc, USE)
        row = {"grade": g, "anchor_design": int(a["design_rank"]), "binder": a["binder_kg_m3"],
               "sp_pct": a["sp_pct_binder"], "flyash_share": a["dv_flyash_frac"],
               "slump_info_bits": inf["bits"], "trial1_cost": float(sc["cost_inr_m3"].iloc[0]),
               "trial1_P_strength": float(sc["P_strength"].iloc[0])}
        CHOICE.append(row)
        key = (round(inf["bits"], 3), -row["trial1_cost"])
        if best is None or key > best[0]:
            best = (key, sc, mixes, a)
    _, sc, mixes, a = best
    TRIALS[g] = sc.assign(anchor_design=int(a["design_rank"]))
    MIXES[g] = mixes
CHOICE = pd.DataFrame(CHOICE)
print("slump information of each possible trial set (higher = teaches more):")
print(CHOICE.to_string(index=False, float_format=F3))
for g in GRADES:
    t = TRIALS[g]
    print(f"\n{g}: trials built on design {t['anchor_design'].iloc[0]}")
    print(t[["trial_no", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3", "admixture_kg_m3",
             "sp_pct_binder", "wb_ratio", "wc_counted", "fc_pred", "P_strength", "slump_pred", "P_slump",
             "inside_training_range"]].to_string(index=False, float_format=F3))

## 6. Optional slump-learning mixes (up to 3 per grade)

`[BO]` From every legal mix (in range or not), greedily add the one that adds the most slump
information **on top of** the 3 trials, skipping mixes with less than an even chance on strength
(no point casting cubes that will surely fail) and anything already handed out. Expert review
before casting — these are for learning, not supply.

In [ ]:
KEY = ["dv_binder_kg_m3", "dv_flyash_frac", "dv_ggbs_frac", "dv_sp_pct_binder"]
_k = lambda df: set(df[KEY].round(6).astype(str).agg("|".join, axis=1))
LEARN = {}
for g in GRADES:
    pool = EXPL[g][(EXPL[g]["gate_decided"].astype(bool)) & (EXPL[g]["P_strength"] >= 0.5)].copy()
    pool = pool[~pool[KEY].round(6).astype(str).agg("|".join, axis=1).isin(_k(DESIGNS[g]) | _k(TRIALS[g]))]
    pool = SEL.price_and_carbon(pool.reset_index(drop=True))
    chosen, have = [], TRIALS[g].copy()
    base_bits = information(wmodel, have)["bits"]
    for _ in range(3):
        if pool.empty:
            break
        gains = [information(wmodel, pd.concat([have, pool.iloc[[i]]], ignore_index=True))["bits"] - base_bits
                 for i in range(len(pool))]
        i = int(np.argmax(gains))
        if gains[i] < 1e-3:
            break
        pick = pool.iloc[[i]].assign(extra_slump_bits=gains[i])
        chosen.append(pick)
        have = pd.concat([have, pool.iloc[[i]]], ignore_index=True)
        base_bits += gains[i]
        pool = pool.drop(pool.index[i]).reset_index(drop=True)
    LEARN[g] = (pd.concat(chosen, ignore_index=True).assign(grade=g, learn_no=lambda d: range(1, len(d) + 1))
                if chosen else pd.DataFrame())
    if len(LEARN[g]):
        print(f"\n{g}: optional slump-learning mixes")
        print(LEARN[g][["learn_no", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3", "sp_pct_binder",
                        "wb_ratio", "fc_pred", "P_strength", "slump_pred", "inside_training_range",
                        "extra_slump_bits"]].to_string(index=False, float_format=F3))

## 7. How much will this round teach? (worked out before casting)

In [ ]:
LEARNING = []
for g in GRADES:
    t = TRIALS[g]
    a = information(wmodel, t, USE)
    full = pd.concat([t, LEARN[g]], ignore_index=True) if len(LEARN[g]) else t
    b = information(wmodel, full, USE)
    st = information(smodel, t) if smodel is not None else {"bits": float("nan")}
    LEARNING.append({"grade": g, "slump_bits_3_trials": a["bits"], "slump_bits_with_optional": b["bits"],
                     "slump_sd_now_mm": a["sd_before"], "slump_sd_after_trials_mm": a["sd_after"],
                     "slump_sd_after_all_mm": b["sd_after"], "strength_bits_3_trials": st["bits"]})
allt = pd.concat([TRIALS[g] for g in GRADES] + [LEARN[g] for g in GRADES if len(LEARN[g])], ignore_index=True)
allinf = information(wmodel, allt, USE)
LEARNING.append({"grade": "both", "slump_bits_3_trials": information(wmodel, pd.concat([TRIALS[g] for g in GRADES]), USE)["bits"],
                 "slump_bits_with_optional": allinf["bits"], "slump_sd_now_mm": allinf["sd_before"],
                 "slump_sd_after_trials_mm": information(wmodel, pd.concat([TRIALS[g] for g in GRADES]), USE)["sd_after"],
                 "slump_sd_after_all_mm": allinf["sd_after"],
                 "strength_bits_3_trials": information(smodel, pd.concat([TRIALS[g] for g in GRADES]))["bits"]})
LEARNING = pd.DataFrame(LEARNING)
print(LEARNING.to_string(index=False, float_format=F3))
print("\nsd = the model's own uncertainty about its slump correction over the mixes it will be asked about.")
print("[PC] The screen still adds a floor of", f"{wmodel['out_of_group_sd_mm']:.3f}",
      "mm (notebook 10 leave-one-group-out error) when it judges a mix, so P_slump stays cautious.")

## 8. The single best next experiment (industry view)

Rule `[PC]`: IS 10262 cl 5.8 says Trial 1 is cast first — it fixes the water for Trials 3 and 4.
So, for each grade in `BEST_GRADES`, the best next experiment is Trial 1 of its chosen set, **if** it passes the IS gate, sits
inside what the models have seen, and has P_strength ≥ 0.90 and P_slump ≥ 0.80. Otherwise the
pool (designs + trials + learning mixes) is filtered by those same tests and the one that teaches
the slump model most wins, cheapest first on ties.

In [ ]:
def _ok(df):
    return df[df["gate_decided"].astype(bool) & (df["inside_training_range"] == True)
              & (df["P_strength"] >= 0.90) & (df["P_slump"] >= 0.80)]
BEST_ROWS = []
for g in BEST_GRADES:
    t1 = TRIALS[g][TRIALS[g]["trial_no"] == 1]
    if len(_ok(t1)):
        BEST = t1.iloc[0].copy(); why = f"Trial 1 of the chosen {g} set (IS 10262 cl 5.8: cast first)"
        BEST_MIX = MIXES[g][0]
    else:
        pool = _ok(pd.concat([DESIGNS[g], TRIALS[g], LEARN[g]], ignore_index=True)).copy()
        if len(pool):
            pool["bits"] = [information(wmodel, pool.iloc[[i]])["bits"] for i in range(len(pool))]
            BEST = pool.sort_values(["bits", "cost_inr_m3"], ascending=[False, True]).iloc[0].copy()
            why = f"most slump information among supply-ready {g} mixes"
        else:
            # [PC] no supply-ready mix: the next experiment is the planned trial most likely to reach the
            #      target mean -- a LEARNING experiment that fixes the strength line, not a supply mix
            pool = TRIALS[g][TRIALS[g]["gate_decided"].astype(bool)].copy()
            BEST = pool.sort_values(["P_strength", "cost_inr_m3"], ascending=[False, True]).iloc[0].copy()
            why = (f"NO supply-ready {g} mix (none in range with P_strength >= 0.90 and P_slump >= 0.80): "
                   f"Trial {int(BEST['trial_no'])}, the planned trial most likely to reach the target -- learning experiment")
        _rq = replace(REQ[g], water_reduction_pct=float(WR.water_reduction_pct(float(BEST["dv_sp_pct_binder"]), wmodel["water_reduction_fit"])))
        BEST_MIX = MIXES[g][list(TRIALS[g].index).index(BEST.name)] if why.startswith("NO supply") else L0.design_mix(_rq, STATE[g], binder_kg_m3=float(BEST["dv_binder_kg_m3"]),
                                 flyash_frac=float(BEST["dv_flyash_frac"]), ggbs_frac=float(BEST["dv_ggbs_frac"]),
                                 admixture_pct_of_binder=float(BEST["dv_sp_pct_binder"]))
    ca20 = ca10 = None
    if STATE[g].ca_split_20_10:
        s20, s10 = STATE[g].ca_split_20_10
        ca20 = BEST_MIX.coarse_agg_kg_m3 * s20 / (s20 + s10); ca10 = BEST_MIX.coarse_agg_kg_m3 - ca20
    BEST_ROWS.append({
        "grade": g, "why": why, "cement": STATE[g].cement_type, "use": REQ[g].use.name, "exposure": REQ[g].exposure.name,
        "cement_kg_m3": BEST_MIX.cement_kg_m3, "flyash_kg_m3": BEST_MIX.flyash_kg_m3,
        "binder_kg_m3": BEST_MIX.binder_kg_m3, "free_water_kg_m3": BEST_MIX.water_kg_m3,
        "admixture_kg_m3": BEST_MIX.admixture_kg_m3, "sp_pct_binder": float(BEST["sp_pct_binder"]),
        "coarse_20mm_ssd_kg_m3": ca20, "coarse_10mm_ssd_kg_m3": ca10,
        "coarse_total_ssd_kg_m3": BEST_MIX.coarse_agg_kg_m3, "crushed_sand_ssd_kg_m3": BEST_MIX.fine_agg_kg_m3,
        "w_binder": BEST_MIX.wc_ratio, "w_c_counted_IS456": BEST_MIX.wc_counted,
        "target_mean_mpa": float(BEST["target_mean"]), "pred_28d_mpa": float(BEST["fc_pred"]),
        "pred_28d_sd_mpa": float(BEST["fc_sd"]), "P_28d_meets_target": float(BEST["P_strength"]),
        "target_slump_mm": REQ[g].target_slump_mm, "pred_slump_mm": float(BEST["slump_pred"]),
        "pred_slump_lo90": float(BEST["slump_lo90"]), "pred_slump_hi90": float(BEST["slump_hi90"]),
        "P_slump_in_tolerance": float(BEST["P_slump"]),
        "slump_info_bits": information(wmodel, pd.DataFrame([BEST]))["bits"],
        "cost_inr_m3_placeholder_rates": float(BEST["cost_inr_m3"]),
        "inside_training_range": bool(BEST["inside_training_range"]), "gate_decided": bool(BEST["gate_decided"]),
        "validation_level": "COMPUTATIONALLY_COMPLIANT"})
for row in BEST_ROWS:
    print(f"\n--- best next {row['grade']} ---")
    for k, v in row.items():
        print(f"  {k:32s} {v:12.3f}" if isinstance(v, float) else f"  {k:32s} {v}")

## 8b. How lean could the IS code alone go? (outside the AI models' range)

The AI models only know mixes from about 327 kg/m³ binder up, so every design above sits there,
and M10 and M15 come out as the **same** mix. The IS gate alone (IS 456 Table 5 PCC mild: min
cement 220 kg/m³, max free w/c 0.60; IS 10262 water at 130 mm) allows leaner mixes. Below is the
leanest legal mix per grade, with the IS 10262 Fig 1 strength `[IS]`/`[EI]` (digitised curve, PPC)
and the AI numbers **flagged as outside range**. It is added to the plan as an **optional** learning cast (engineer approval first): a measured result
there teaches both models the lean region M10/M15 really need.

In [ ]:
_wfit = wmodel["water_reduction_fit"]
LEAN = []
for g in GRADES:
    found = None
    for b in range(220, 331, 5):
        for sp in np.round(np.linspace(ADMIXTURE_RANGE_PCT[0], ADMIXTURE_RANGE_PCT[1], 7), 3)[::-1]:
            rq = replace(REQ[g], water_reduction_pct=float(WR.water_reduction_pct(float(sp), _wfit)))
            for fa in (0.0, 0.05, 0.10, 0.125):
                try:
                    m = L0.design_mix(rq, STATE[g], binder_kg_m3=float(b), flyash_frac=fa, admixture_pct_of_binder=float(sp))
                except Exception:
                    continue
                if m.decided:
                    found = m; break
            if found: break
        if found: break
    if found is None:
        print(f"{g}: no decided mix below 330 kg/m3"); continue
    sc, _ = SC.screen(REQ[g], STATE[g], wmodel=wmodel, smodel=smodel, support=SUP, mode="score", mixes=[found])
    r = sc.iloc[0]
    _row = r.copy()
    _row["learn_no"] = (int(LEARN[g]["learn_no"].max()) + 1) if len(LEARN[g]) else 1
    _row["extra_slump_bits"] = float("nan")
    _row["learn_role"] = "optional LEAN IS-legal mix (outside AI range, engineer approval first)"
    _row["grade"] = g
    LEARN[g] = pd.concat([LEARN[g], pd.DataFrame([_row])], ignore_index=True)
    LEAN.append({"grade": g, "binder_kg_m3": found.binder_kg_m3, "cement_kg_m3": found.cement_kg_m3,
                 "flyash_kg_m3": found.flyash_kg_m3, "water_kg_m3": found.water_kg_m3,
                 "sp_pct_binder": float(r["sp_pct_binder"]), "wb_ratio": found.wc_ratio, "wc_counted": found.wc_counted,
                 "fig1_28d_mpa": float(SR.fig1_strength(found.wc_ratio, "PPC")), "target_mean": float(r["target_mean"]),
                 "ai_fc_pred": float(r["fc_pred"]), "ai_slump_pred": float(r["slump_pred"]),
                 "inside_training_range": bool(r["inside_training_range"]),
                 "binder_saved_vs_best_kg_m3": 330.0 - found.binder_kg_m3})
LEAN = pd.DataFrame(LEAN)
print(LEAN.round(3).to_string(index=False))
print("added to the plan as an OPTIONAL learning cast (engineer approval first); also in the lab pack, sheet 'Leanest IS mix'")

## 9. Lab batch sheets (0.050 m³, moisture-corrected) and the results log

`[IS]` Weights per IS 10262 Annex A-10 / A-11; batch size covers 6 cubes + slump + density
(`is_rules.trial_batch_volume`). Moisture: the casting day's value goes in `MOISTURE_PCT`
(section 1); until then the training plant's median is used **[ASSUMED]**.

In [ ]:
VOL = IS.trial_batch_volume(6, msa_mm=STATE[GRADES[0]].msa_mm)["batch_m3"]
RUN = 1
if not LOG.empty and "round" in LOG:
    last = int(pd.to_numeric(LOG["round"], errors="coerce").max())
    filled = LOG[pd.to_numeric(LOG["round"], errors="coerce") == last]["outcome"].fillna("").astype(str).str.strip().ne("").any()
    RUN = last + 1 if filled else last
CASTS, BATCH, SEEN = [], [], {}
def _mix_for(g, r):
    rq = replace(REQ[g], water_reduction_pct=float(WR.water_reduction_pct(float(r["dv_sp_pct_binder"]), wmodel["water_reduction_fit"])))
    return L0.design_mix(rq, STATE[g], binder_kg_m3=float(r["dv_binder_kg_m3"]), flyash_frac=float(r["dv_flyash_frac"]),
                         ggbs_frac=float(r["dv_ggbs_frac"]), admixture_pct_of_binder=float(r["dv_sp_pct_binder"]))
for g in GRADES:
    items = [(f"T{int(r.trial_no)}", "IS 10262 cl 5.8 trial", r, MIXES[g][i]) for i, (_, r) in enumerate(TRIALS[g].iterrows())]
    if len(LEARN[g]):
        items += [(f"L{int(r.learn_no)}", (r["learn_role"] if "learn_role" in r and isinstance(r["learn_role"], str) else "optional slump-learning mix"), r, _mix_for(g, r)) for _, r in LEARN[g].iterrows()]
    sm = MOISTURE_PCT["sand"] if MOISTURE_PCT["sand"] is not None else FAM[g]["plant"]["sand_moisture_median"]
    cm = MOISTURE_PCT["ca"] if MOISTURE_PCT["ca"] is not None else FAM[g]["plant"]["ca_moisture_median"]
    for tag, role, r, m in items:
        tid = f"PPC-{g}-R{RUN}-{tag}"
        assert abs(m.water_kg_m3 - float(r["water_kg_m3"])) < 1e-6, f"{tid}: rebuilt mix differs from the scored one"
        # [PC] the same mix planned for two grades is ONE physical batch, judged against both targets
        sig = tuple(round(float(v), 3) for v in (m.cement_kg_m3, m.flyash_kg_m3, m.water_kg_m3, m.admixture_kg_m3,
                                                 m.fine_agg_kg_m3, m.coarse_agg_kg_m3))
        if sig in SEEN:
            CASTS[SEEN[sig]]["serves_grades"] += f";{g}"
            CASTS[SEEN[sig]]["note"] += f" Same mix as {g} {tag} -> one batch serves both."
            continue
        SEEN[sig] = len(CASTS)
        cubes = "3 x 7 d + 3 x 28 d" if role.startswith("IS") else "3 x 28 d"
        CASTS.append({"trial_id": tid, "round": RUN, "grade": g, "serves_grades": g, "note": "", "role": role, "cubes": cubes,
                      "cement_type": STATE[g].cement_type, "target_slump_mm": REQ[g].target_slump_mm,
                      "target_mean_mpa": float(r["target_mean"]),
                      "cement_kg_m3": m.cement_kg_m3, "flyash_kg_m3": m.flyash_kg_m3, "water_kg_m3": m.water_kg_m3,
                      "admixture_kg_m3": m.admixture_kg_m3, "binder_kg_m3": m.binder_kg_m3,
                      "coarse_agg_kg_m3": m.coarse_agg_kg_m3, "fine_agg_kg_m3": m.fine_agg_kg_m3,
                      "wb_ratio": m.wc_ratio, "wc_counted": m.wc_counted, "msa": STATE[g].msa_mm,
                      **{f: float(r[f]) for f in sorted(set(wmodel["features"]) | set(smodel["features"]))},
                      "slump_pred": float(r["slump_pred"]), "slump_lo90": float(r["slump_lo90"]),
                      "slump_hi90": float(r["slump_hi90"]), "P_slump": float(r["P_slump"]),
                      "fc_pred": float(r["fc_pred"]), "P_strength": float(r["P_strength"]),
                      "inside_training_range": bool(r["inside_training_range"]),
                      "slump_info_bits": information(wmodel, pd.DataFrame([r]))["bits"],
                      "validation_level": "COMPUTATIONALLY_COMPLIANT"})
        bs = L0.batch_sheet(m, STATE[g], volume_m3=VOL, sand_moisture_pct=sm, ca_moisture_pct=cm, round_kg=0.001)
        _split = STATE[g].ca_split_20_10
        _c20 = m.coarse_agg_kg_m3 * _split[0] / sum(_split) if _split else None
        _full = {"cement": m.cement_kg_m3, "fly ash": m.flyash_kg_m3, "water": m.water_kg_m3,
                 "admixture": m.admixture_kg_m3, "fine aggregate": m.fine_agg_kg_m3,
                 "coarse 20-10": _c20, "coarse 10-4.75": (m.coarse_agg_kg_m3 - _c20) if _split else None}
        for b in bs["rows"]:
            for _k, _v in _full.items():          # full-precision design SSD (batch_sheet echoes 2 dp)
                if str(b.get("material", "")).startswith(_k) and _v is not None:
                    b = {**b, "design_ssd_kg_m3": _v}
            BATCH.append({"trial_id": tid, "grade": g, **b, "batch_m3": VOL,
                          "sand_moisture_pct_used": sm, "ca_moisture_pct_used": cm,
                          "moisture_source": "casting day" if MOISTURE_PCT["sand"] is not None else "[ASSUMED] training-plant median"})
CASTS, BATCH = pd.DataFrame(CASTS), pd.DataFrame(BATCH)
print(f"round R{RUN}: {len(CASTS)} casts planned ({(CASTS['role'].str.startswith('IS')).sum()} IS trials, "
      f"{(~CASTS['role'].str.startswith('IS')).sum()} optional learning mixes); batch {VOL:.3f} m3 each")

# the results log: append the new plan, keep every row the lab has already filled
new = CASTS.copy()
for c in LAB_COLS:
    new[c] = None
if LOG.empty:
    out = new
else:
    keep = LOG[~((pd.to_numeric(LOG["round"], errors="coerce") == RUN) & LOG["outcome"].fillna("").astype(str).str.strip().eq(""))]
    new = new[~new["trial_id"].isin(set(keep["trial_id"]))]
    out = pd.concat([keep, new], ignore_index=True)
num = out.select_dtypes("number").columns
out[num] = out[num].round(3)
LOGX_DF = out
if Path(LOG_PATH).exists():
    out.to_csv(LOG_PATH, index=False)
    print(f"{LOG_PATH}: {len(out)} rows ({len(new)} new planned rows; filled rows never touched)")
else:
    print(f"results log: {len(out)} planned rows -> sheet 'Results log' in the lab pack (the lab saves it as {LOG_PATH})")

## 10. How the lab must test, so every result teaches the slump model

1. **Batch exactly from the sheet**; record what was actually weighed and the day's aggregate
   moisture (IS 2386 Part 3). Use the same PPC, fly ash and crushed sand as listed.
2. **Do not adjust water or admixture before the 15-minute slump.** A reading after an
   adjustment belongs to a different mix.
3. **True slump** (IS 1199 Part 2) at **15 min** after batching — the time the AI model's data
   were taken (10–25 min, median 15) `[PC]` — then at **45 min** and **60 min** (delivery
   readings). Record concrete temperature each time (IS 1199 Part 1 cl 6.4) and the slump shape
   (shear / collapse is not a valid slump, IS 1199 Part 2 cl 4.3.4).
4. If you correct the slump (IS 10262 cl 5.8 = Trial 2), log the corrected batch as a **new row**
   with its own readings and `actual_free_water_kg_m3`. Never overwrite the first.
5. Cubes: 3 at 7 d + 3 at 28 d for each IS trial; 3 at 28 d for each learning mix
   (IS 516 Part 1 Sec 1). A sample = mean of 3 cubes (IS 456 cl 15.3 / 15.4).
6. Write `measured` in `outcome` when you have readings. If a mix was not cast, write why
   (`human_rejected_before_cast`, `material_unavailable`, `cost_rejected`,
   `constructability_rejected`, `other_project_constraint`, `experiment_failed`) and leave the
   numbers blank. Keep failures — they teach too.
7. Save the sheet as `M10_M15_PPC_results_log.csv` in this folder and re-run this notebook.

In [ ]:
PROTO = pd.DataFrame({"step": [
    "Batch exactly from the sheet; record actual weights and the day's aggregate moisture (IS 2386 Part 3). Same PPC, fly ash and crushed sand as listed.",
    "Do NOT adjust water or admixture before the 15-minute slump reading.",
    "True slump (IS 1199 Part 2) at 15 min after batching (the AI model's data timing, [PC]), then 45 min and 60 min.",
    "Record concrete temperature at every reading (IS 1199 Part 1 cl 6.4) and the slump shape (shear/collapse invalid, IS 1199 Part 2 cl 4.3.4).",
    "If you correct the slump (IS 10262 cl 5.8, Trial 2), log the corrected batch as a NEW row with actual_free_water_kg_m3.",
    "Cubes: 3 x 7 d + 3 x 28 d for each IS trial, 3 x 28 d for each learning mix (IS 516 Part 1 Sec 1); sample = mean of 3 (IS 456 cl 15.3/15.4).",
    "outcome = measured when you have readings; otherwise write the reason word and leave numbers blank. Keep failures.",
    "Save as M10_M15_PPC_results_log.csv in this folder and re-run notebook 19: the AI models learn and plan the next round."]})

def r3(df):
    df = df.copy(); n = df.select_dtypes("number").columns; df[n] = df[n].round(3); return df
DES = r3(pd.concat([DESIGNS[g] for g in GRADES], ignore_index=True)[
    ["grade", "design_rank", "design_status", "cement_kg_m3", "flyash_kg_m3", "binder_kg_m3", "water_kg_m3",
     "admixture_kg_m3", "sp_pct_binder", "coarse_agg_kg_m3", "fine_agg_kg_m3", "wb_ratio", "wc_counted",
     "target_mean", "fc_pred", "fc_sd", "P_strength", "slump_pred", "slump_lo90", "slump_hi90", "P_slump",
     "inside_training_range", "cost_inr_m3", "carbon_kg_m3", "cost_status", "validation_level"]])
TRI = r3(pd.concat([TRIALS[g] for g in GRADES], ignore_index=True)[
    ["grade", "anchor_design", "trial_no", "trial_role", "cement_kg_m3", "flyash_kg_m3", "binder_kg_m3",
     "water_kg_m3", "admixture_kg_m3", "sp_pct_binder", "coarse_agg_kg_m3", "fine_agg_kg_m3", "wb_ratio",
     "wc_counted", "target_mean", "fc_pred", "P_strength", "slump_pred", "slump_lo90", "slump_hi90", "P_slump",
     "inside_training_range", "cost_inr_m3", "trial_note"]])
LRN = r3(pd.concat([LEARN[g] for g in GRADES if len(LEARN[g])], ignore_index=True)[
    ["grade", "learn_no", "cement_kg_m3", "flyash_kg_m3", "binder_kg_m3", "water_kg_m3", "admixture_kg_m3",
     "sp_pct_binder", "coarse_agg_kg_m3", "fine_agg_kg_m3", "wb_ratio", "wc_counted", "fc_pred", "P_strength",
     "slump_pred", "inside_training_range", "extra_slump_bits"]]) if any(len(LEARN[g]) for g in GRADES) else pd.DataFrame()
BESTDF = r3(pd.DataFrame(BEST_ROWS))
BAT = r3(BATCH); LOGX = LOGX_DF; DOMX = r3(DOM); LRNG = r3(LEARNING); CHO = r3(CHOICE)

# One CSV per grade, written for people: row 1 = plain-English header, row 2 = what it means
# (with this file's own number as the example), row 3 = the values (3 decimals).
READABLE = [
    ("grade", "Concrete grade", "{v} = characteristic 28-day cube strength of {fck} MPa (IS 456)"),
    ("why", "Why this mix was chosen", "How the SDL picked this mix as the next one to cast"),
    ("cement", "Cement type", "{v} = Portland Pozzolana Cement, IS 1489 (Part 1)"),
    ("use", "Use: RCC or PCC", "{v}: RCC = reinforced concrete, PCC = plain concrete (no steel)"),
    ("exposure", "Exposure condition", "{v} exposure under IS 456; it sets the minimum cement and maximum w/c"),
    ("cement_kg_m3", "PPC (kg per m3)", "Weigh {v} kg of PPC for every 1 m3 of concrete"),
    ("flyash_kg_m3", "Added fly ash (kg per m3)", "{v} kg of fly ash added on top of the PPC, per m3"),
    ("binder_kg_m3", "Total binder = PPC + fly ash (kg per m3)", "{v} kg per m3 of PPC and fly ash together"),
    ("free_water_kg_m3", "Free water (kg per m3)", "{v} kg per m3 of water free to react; adjust for the aggregate moisture on the day"),
    ("admixture_kg_m3", "Superplasticiser (kg per m3)", "{v} kg per m3 of PCE superplasticiser (IS 9103), weighed separately"),
    ("sp_pct_binder", "Superplasticiser dose (% of binder)", "{v} % of the total binder mass"),
    ("coarse_20mm_ssd_kg_m3", "Coarse aggregate 20-10 mm, SSD (kg per m3)", "{v} kg per m3 of 20 mm stone, saturated surface-dry basis"),
    ("coarse_10mm_ssd_kg_m3", "Coarse aggregate 10-4.75 mm, SSD (kg per m3)", "{v} kg per m3 of 10 mm stone, saturated surface-dry basis"),
    ("coarse_total_ssd_kg_m3", "Total coarse aggregate, SSD (kg per m3)", "{v} kg per m3 = 20 mm + 10 mm stone together"),
    ("crushed_sand_ssd_kg_m3", "Crushed sand, SSD (kg per m3)", "{v} kg per m3 of crushed sand, saturated surface-dry basis"),
    ("w_binder", "Water / total binder ratio", "{v} kg of free water for every 1 kg of PPC + fly ash"),
    ("w_c_counted_IS456", "Water / cement ratio checked against IS 456 Table 5", "{v}: water / (PPC + the part of the fly ash IS 456 allows to count); must stay at or below the Table 5 limit"),
    ("target_mean_mpa", "Target mean 28-day strength (MPa)", "The 28-day average of 3 cubes should reach {v} MPa (IS 10262 cl 4.2)"),
    ("pred_28d_mpa", "Predicted 28-day strength (MPa)", "The AI model expects about {v} MPa (mean of 3 cubes); a forecast, not a test result"),
    ("pred_28d_sd_mpa", "Uncertainty of predicted strength, 1 standard deviation (MPa)", "Predicted strength is about plus or minus {v} MPa"),
    ("P_28d_meets_target", "Chance 28-day strength reaches the target (0 to 1)", "{v} = the model's chance the cubes reach the target mean"),
    ("target_slump_mm", "Target slump (mm)", "Aim for {v} mm slump at the 15-minute reading (accepted band +/- 25 mm, IS 4926 cl 6.2.1)"),
    ("pred_slump_mm", "Predicted slump (mm)", "The AI model expects about {v} mm"),
    ("pred_slump_lo90", "Predicted slump, low end of 90 % range (mm)", "9 times in 10 the slump should be above {v} mm"),
    ("pred_slump_hi90", "Predicted slump, high end of 90 % range (mm)", "9 times in 10 the slump should be below {v} mm"),
    ("P_slump_in_tolerance", "Chance slump lands within +/- 25 mm of target (0 to 1)", "{v} = the model's chance the slump is on target"),
    ("slump_info_bits", "How much this test teaches the slump model (bits)", "{v} bits; higher = the result improves the slump model more"),
    ("cost_inr_m3_placeholder_rates", "Indicative cost (INR per m3, placeholder rates)", "About Rs {v} per m3 using placeholder prices, not real quotes"),
    ("inside_training_range", "Inside the AI models' experience?", "{v}: True = similar mixes are in the training data; False = trust the predictions less"),
    ("gate_decided", "Passed every IS code check?", "{v}: True = all IS 456 / IS 10262 checks passed"),
    ("validation_level", "Validation stage", "{v} = meets the codes on paper only; not yet cast or tested"),
]

def _fmt(v):
    if isinstance(v, (float, np.floating)) and not isinstance(v, bool):
        return f"{float(v):.3f}"
    return str(v)

def write_readable(row, path):
    import csv
    keys = [k for k, _, _ in READABLE if k in row.index]
    extra = [k for k in row.index if k not in keys]
    hdr = [h for k, h, _ in READABLE if k in row.index] + extra
    fck = IS.target_mean_strength(str(row["grade"]))["fck"]
    expl = [t.format(v=_fmt(row[k]), fck=f"{fck:.0f}") for k, _, t in READABLE if k in row.index] + ["" for _ in extra]
    vals = [_fmt(row[k]) for k in keys + extra]
    with open(path, "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.writer(fh)
        w.writerow(hdr); w.writerow(expl); w.writerow(vals)

for _g, _b in BESTDF.groupby("grade"):
    write_readable(_b.iloc[0], f"19_next_experiment_{_g}.csv")
    print(f"wrote 19_next_experiment_{_g}.csv (readable headers + meaning row)")
# everything else lives in the lab pack only (one file for the lab, a clean folder)
Path("19_M10_M15_PPC_model_state.json").write_text(json.dumps({
    "round": RUN, "cement": "PPC", "materials_from": {g: FAM[g]["family"] for g in GRADES},
    "slump_model": {"rows": wmodel["n"], "updated_with": wmodel.get("updated_with", [])},
    "strength_model": {"rows": smodel["n"], "updated_with": smodel.get("updated_with", [])},
    "lab_batches_used": {"slump": UPD["n_slump_batches"], "strength": UPD["n_strength_batches"]},
    "learning_this_round": LRNG.to_dict("records"),
    "assumed": ASSUMED},
    indent=2, default=str), encoding="utf-8")

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
wb = Workbook(); wb.remove(wb.active)
YEL = PatternFill("solid", fgColor="FFF2A8")
def sheet(name, df, title, yellow=()):
    ws = wb.create_sheet(name)
    ws.cell(1, 1, title).font = Font(bold=True, size=12)
    for j, c in enumerate(df.columns, 1):
        ws.cell(3, j, str(c)).font = Font(bold=True)
        ws.cell(3, j).alignment = Alignment(wrap_text=True)
        ws.column_dimensions[ws.cell(3, j).column_letter].width = max(12, min(40, len(str(c)) + 2))
    for i, row in enumerate(df.itertuples(index=False), 4):
        for j, v in enumerate(row, 1):
            v = None if (isinstance(v, float) and math.isnan(v)) else v
            cell = ws.cell(i, j, v.item() if hasattr(v, "item") else v)
            if isinstance(v, float): cell.number_format = "0.000"
            if df.columns[j - 1] in yellow: cell.fill = YEL
    ws.freeze_panes = "A4"
README = pd.DataFrame({"read me": README_LINES})
sheet("Read me", README, README_TITLE)
sheet("Best next", BESTDF.set_index("grade").T.reset_index().rename(columns={"index": "item"}), "Single best next experiment per grade")
sheet("Trials", TRI, "3 IS 10262 cl 5.8 trials per grade -- chosen to teach the slump model")
sheet("Designs", DES, "5 designs per grade (cost = PLACEHOLDER rates)")
if len(LRN): sheet("Slump-learning", LRN, "Optional slump-learning mixes -- expert review before casting")
sheet("Batch sheets", BAT, f"What to weigh -- {VOL:.3f} m3 per batch, kg to 3 decimals")
sheet("Results log", LOGX, "Lab fills the yellow cells -> save as M10_M15_PPC_results_log.csv -> re-run notebook 19", yellow=LAB_COLS)
sheet("How to test", PROTO, "Test protocol -- so every result teaches the slump model")
sheet("Learning", LRNG, "How much this round will teach (worked out before casting)")
sheet("Domain check", DOMX, "Is each order inside what the AI models have seen?")
if "LEAN" in globals() and len(LEAN):
    sheet("Leanest IS mix", r3(LEAN), "Leanest mix the IS code alone allows (outside the AI range) -- information + optional cast")
wb.save("19_M10_M15_PPC_lab_pack.xlsx")
print("wrote 19_M10_M15_PPC_model_state.json and 19_M10_M15_PPC_lab_pack.xlsx")